Regresión Lineal Multivariante - EJECUTAR TODO, PARA LAS METRICAS COMPLETAS

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pandas.api.types import is_datetime64_any_dtype

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas Académicas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- PARÁMETROS MODIFICADOS ---
N_LAGS_AQI = 120
N_LAGS_EXO = 120

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5

# --- Parámetros de Ploteo ---
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')
SIMULATED_NANS_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLineal_full'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_CSVS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predicciones_csv')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_CSVS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)

# --- MÉTODOS A PROCESAR (CORREGIDO) ---
METHODS_TO_PROCESS = [
    'Linear Interpolation',
    'LSTM-VAE',
    'MOMENT Finetuned',
    'MOMENT Pretrained',
    'Random Forest',
    'SARIMA'
]

# --- Funciones Auxiliares (sin cambios) ---
def create_multivariate_lagged_features(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    df_copy = df_full_city_data.copy()
    features_list = []
    for i in range(horizon, n_lags_target + horizon):
        features_list.append(df_copy[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(horizon, n_lags_exo + horizon):
            features_list.append(df_copy[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_features_cols:
        if tf_col in df_copy.columns:
            features_list.append(df_copy[tf_col])
    df_features = pd.concat(features_list, axis=1)
    return df_features

def generate_multi_step_forecasting_plots(df_imputed_data, df_ground_truth_data, model, X_all_features, original_nan_indices, city, scenario, method, rep_id, horizon, output_dir, n_examples=1, history_len=300):
    max_lags = max(N_LAGS_AQI, N_LAGS_EXO)
    all_possible_points = range(max_lags + history_len, len(df_imputed_data) - horizon - 1)
    valid_split_points = []
    if not original_nan_indices:
        valid_split_points = list(all_possible_points)
    else:
        for p in all_possible_points:
            future_window = range(p, p + horizon)
            history_window = range(p - max_lags, p)
            is_future_clean = not any(idx in original_nan_indices for idx in future_window)
            is_history_dirty = any(idx in original_nan_indices for idx in history_window)
            if is_future_clean and is_history_dirty:
                valid_split_points.append(p)
    if len(valid_split_points) < n_examples: return
    example_indices = random.sample(valid_split_points, n_examples)
    for i, split_point in enumerate(example_indices):
        history = df_imputed_data.iloc[:split_point]
        ground_truth_df = df_ground_truth_data.iloc[split_point : split_point + horizon]
        X_forecast_features = X_all_features.loc[ground_truth_df.index]
        if X_forecast_features.empty or X_forecast_features.isnull().values.any(): continue
        forecast_values = model.predict(X_forecast_features)
        lower_bound = history[TARGET_VARIABLE].quantile(0.01); upper_bound = history[TARGET_VARIABLE].quantile(0.99)
        padding = (upper_bound - lower_bound) * 0.1
        clipped_forecast_values = np.clip(forecast_values, lower_bound - padding, upper_bound + padding)
        forecast_df = pd.DataFrame({'Forecast': clipped_forecast_values}, index=ground_truth_df.index)
        plt.figure()
        plot_history = history.iloc[-history_len:]; plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(ground_truth_df.index, ground_truth_df[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(forecast_df.index, forecast_df['Forecast'], label='Forecast', color='green', linestyle='--')
        plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f'Linear Regressor Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        scenario_safe_for_path = scenario.replace('/', '_').replace('\\', '_'); plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe_for_path); os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe_for_path}_{rep_id}_h{horizon}.png'; plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150); plt.close()

def train_predict_eval_multi_horizon(df_city_data_imputed, df_city_data_ground_truth, original_nan_indices, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizons_list, n_cv_splits, plot_params=None):
    results_by_horizon = []
    max_lags = max(n_lags_target, n_lags_exo)
    for horizon in tqdm(horizons_list, desc="  Horizons", leave=False):
        X = create_multivariate_lagged_features(df_city_data_imputed, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon)
        y = df_city_data_ground_truth[target_col].rename('target_h')
        df_processed = pd.concat([X, y], axis=1).dropna()
        if df_processed.empty or len(df_processed) < n_cv_splits * 2:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan}); continue
        y_proc = df_processed['target_h']; X_proc = df_processed.drop(columns=['target_h'])
        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        maes, rmses = [], []
        for _, (train_index, test_index) in enumerate(tscv.split(X_proc)):
            X_train, X_test = X_proc.iloc[train_index], X_proc.iloc[test_index]
            y_train, y_test = y_proc.iloc[train_index], y_proc.iloc[test_index]
            if y_test.empty: continue
            target_is_real_mask = ~y_test.index.isin(original_nan_indices)
            if not original_nan_indices:
                history_is_dirty_mask = pd.Series(True, index=y_test.index)
            else:
                history_is_dirty_list = []
                for idx in y_test.index:
                    history_window = range(idx - horizon - max_lags, idx - horizon)
                    history_is_dirty_list.append(any(h_idx in original_nan_indices for h_idx in history_window))
                history_is_dirty_mask = pd.Series(history_is_dirty_list, index=y_test.index)
            final_valid_mask = target_is_real_mask & history_is_dirty_mask
            y_test_valid = y_test[final_valid_mask]
            if y_test_valid.empty: continue
            model = LinearRegression().fit(X_train, y_train)
            predictions = model.predict(X_test)
            predictions_valid = predictions[final_valid_mask.values]
            maes.append(mean_absolute_error(y_test_valid, predictions_valid))
            rmses.append(np.sqrt(mean_squared_error(y_test_valid, predictions_valid)))
        results_by_horizon.append({'Horizon': horizon, 'MAE': np.mean(maes) if maes else np.nan, 'RMSE': np.mean(rmses) if rmses else np.nan})
        if plot_params and maes:
            final_model = LinearRegression().fit(X_proc, y_proc)
            all_features_full_df = create_multivariate_lagged_features(df_city_data_imputed, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon)
            generate_multi_step_forecasting_plots(df_imputed_data=df_city_data_imputed, df_ground_truth_data=df_city_data_ground_truth, model=final_model, X_all_features=all_features_full_df, original_nan_indices=original_nan_indices, city=plot_params['city'], scenario=plot_params['scenario'], method=plot_params['method'], rep_id=plot_params['rep_id'], horizon=horizon, output_dir=plot_params['output_dir'], n_examples=plot_params['n_examples'], history_len=plot_params['history_len'])
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month
    print(f"Original DataFrame '{ORIGINAL_DATA_FILE}' loaded.")
except FileNotFoundError:
    print(f"Error: '{ORIGINAL_DATA_FILE}' not found."); df_original_full = None

if df_original_full is not None:
    print("\n--- Forecasting with Original Data (Upper Bound Baseline) ---")
    method_name_baseline = 'Original_Data_Baseline'
    empty_nan_indices = set()
    for city_name in df_original_full['City'].unique():
        print(f"  City: {city_name}")
        df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
            if col in df_city_ground_truth.columns and df_city_ground_truth[col].isnull().any():
                df_city_ground_truth[col] = df_city_ground_truth[col].ffill().bfill().fillna(df_original_full[col].mean())
        plot_parameters_baseline = {'city': city_name, 'scenario': 'N/A', 'method': method_name_baseline, 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR, 'n_examples': N_PLOT_EXAMPLES, 'history_len': HISTORY_LEN_FOR_PLOT}
        horizon_metrics = train_predict_eval_multi_horizon(df_city_ground_truth, df_city_ground_truth, empty_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_parameters_baseline)
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({'Imputation_Method': method_name_baseline, 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE']})
        if horizon_metrics and not np.isnan(horizon_metrics[0]['MAE']): print(f"    Metrics (h={horizon_metrics[0]['Horizon']}): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}")

    print("\n--- Forecasting with Imputed Data (Selected Methods) ---")
    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root):
            print(f"\n--- SKIPPING Imputation Method: '{method_name}' (Directory not found at {imputed_csvs_root}) ---")
            continue
        
        print(f"\n--- Processing Imputation Method: {method_name} ---")
        city_folders = [f for f in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, f))]
        for city_folder in tqdm(city_folders, desc=f"  Cities in {method_name}"):
            city_path = os.path.join(imputed_csvs_root, city_folder)
            original_city_name = city_folder.replace("_", " ")
            df_city_ground_truth = df_original_full[df_original_full['City'] == original_city_name].copy().reset_index(drop=True)
            for scenario_folder in os.listdir(city_path):
                scenario_path = os.path.join(city_path, scenario_folder)
                if not os.path.isdir(scenario_path): continue
                repetition_files = glob.glob(os.path.join(scenario_path, "*.csv"))
                if not repetition_files: continue
                rep_file_path = repetition_files[0]
                file_name = os.path.basename(rep_file_path)
                rep_file_id_parts = [part for part in file_name.split('_') if 'rep' in part or 'seed' in part]
                rep_file_id = "_".join(rep_file_id_parts) if rep_file_id_parts else file_name
                
                nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv"
                try:
                    nan_file_path = os.path.join(SIMULATED_NANS_INPUT_DIR, city_folder, scenario_folder, nan_filename)
                    df_with_nans = pd.read_csv(nan_file_path)
                    original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
                except FileNotFoundError:
                    print(f"        WARNING: Corresponding NaN file not found for '{file_name}'. Skipping impact evaluation checks."); original_nan_indices = set()
                
                # --- BLOQUE DE CARGA DE FECHA SIMPLIFICADO Y CORRECTO ---
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    
                    if 'Date' not in df_rep.columns:
                        raise ValueError("'Date' column not found in CSV.")
                    
                    # 1. Truncar la zona horaria del string de la fecha
                    # Esto convierte '2024-01-01 02:00:00+02:00' en '2024-01-01 02:00:00'
                    df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
                    
                    # 2. Ahora convertir a datetime es trivial y no falla
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'])
                    
                    # 3. Comprobar si las columnas de tiempo ya existen
                    time_cols_exist = all(col in df_rep.columns for col in TIME_FEATURES)
                    
                    # 4. Si no existen, crearlas.
                    if not time_cols_exist:
                        df_rep['Hour'] = df_rep['Date'].dt.hour
                        df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                        df_rep['Month'] = df_rep['Date'].dt.month

                except Exception as e: 
                    print(f"        FATAL ERROR processing file {file_name}: {e}. Skipping."); continue
                # --- FIN DEL BLOQUE ---

                df_city_rep = df_rep[df_rep['City'] == original_city_name].copy().reset_index(drop=True)
                if df_city_rep.isnull().values.any():
                    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
                        if col in df_city_rep.columns and df_city_rep[col].isnull().any(): 
                            df_city_rep[col] = df_city_rep[col].ffill().bfill().fillna(0)
                            
                plot_parameters_rep = {'city': original_city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR, 'n_examples': N_PLOT_EXAMPLES, 'history_len': HISTORY_LEN_FOR_PLOT}
                horizon_metrics_rep = train_predict_eval_multi_horizon(df_city_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_parameters_rep)
                
                for hm_rep in horizon_metrics_rep:
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm_rep['Horizon'], 'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE']})

# --- Bloque final de resumen (sin cambios) ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\nAll detailed prediction metrics saved to: {detailed_output_path}")
    df_summary = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Count=('MAE', 'count')).reset_index()
    for city_name in df_summary['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary[df_summary['City'] == city_name]
        city_metrics_dir = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, city_name_safe)
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_safe}.csv')
        df_city_summary.to_csv(city_summary_path, index=False)
        print(f"  Metrics summary for {city_name} saved to: {city_summary_path}")
    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"\nGlobal average metrics summary saved to: {global_summary_path}")
    print("--- Global Summary by Method and Horizon ---")
    print(df_global_summary.to_string())

script_end_time = time.time()
print(f"\nForecasting script completed. Total time: {(script_end_time - script_start_time)/60:.2f} minutes.")
print(f"Forecast plots saved in: {PREDICTION_PLOTS_OUTPUT_DIR}")

SARIMAX PARALELIZADO

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- PARÁMETROS DE EJECUCIÓN ---
N_CV_SPLITS = 2
HISTORY_LEN_FOR_PLOT = 300
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', 'SARIMA', 'model_orders')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'SARIMAX_pred_full_plots_parallel'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_sarimax_forecasting_plots(df_full_imputed, df_full_ground_truth, model_results, original_nan_indices, city, scenario, method, rep_id, horizon, fold_num, output_dir, history_len=300):
    try:
        plt.figure(figsize=(20, 8))
        forecast_start_index = model_results.model.endog.shape[0]
        if forecast_start_index >= len(df_full_imputed): plt.close(); return
        history = df_full_imputed.iloc[:forecast_start_index]
        if original_nan_indices and method != 'Original_Data_Baseline':
            if not any(idx in original_nan_indices for idx in set(history.index)): plt.close(); return
        future_gt = df_full_ground_truth.iloc[forecast_start_index : forecast_start_index + horizon]
        if original_nan_indices:
            if any(idx in original_nan_indices for idx in set(future_gt.index)): plt.close(); return
        if len(future_gt) < horizon: plt.close(); return
        exog_future = df_full_imputed[EXOGENOUS_COLS + TIME_FEATURES].iloc[forecast_start_index : forecast_start_index + horizon]
        forecast_obj = model_results.get_forecast(steps=horizon, exog=exog_future)
        forecast_df = forecast_obj.summary_frame()
        plot_history_display = history.iloc[-history_len:]
        plt.plot(plot_history_display.index, plot_history_display[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(future_gt.index, future_gt[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(forecast_df.index, forecast_df['mean'], label='Forecast', color='green', linestyle='--')
        plt.axvline(x=history.index[-1], color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        scenario_safe = scenario.replace('/', '_').replace('\\', '_')
        plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
        os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}_fold{fold_num}.png'
        plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
        plt.close()
    except Exception:
        plt.close()

# --- FUNCIÓN DE EVALUACIÓN MODIFICADA PARA PLOTEAR TODOS LOS HORIZONTES ---
def train_predict_eval_sarimax_multi_horizon(df_city_data_input, df_city_data_ground_truth, original_nan_indices, target_col, exo_cols, time_features_cols, sarima_order, sarima_seasonal_order, horizons_list, n_cv_splits, plot_params=None):
    results_by_horizon = []
    y = df_city_data_input[target_col]
    X = df_city_data_input[exo_cols + time_features_cols]
    min_len = (sum(sarima_order) + sum(sarima_seasonal_order)) * 2 + max(horizons_list) + n_cv_splits
    if len(y) < min_len:
        for h in horizons_list: results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon
        
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons_list))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}
    
    for fold, (train_index, test_index) in enumerate(tqdm(tscv.split(X), total=n_cv_splits, desc="    CV Folds", leave=False)):
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        if y_train.empty or X_train.empty: continue
        
        results_fit = None # Inicializar
        try:
            model = SARIMAX(y_train, exog=X_train, order=sarima_order, seasonal_order=sarima_seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
            results_fit = model.fit(disp=False, maxiter=200)
            forecasts = results_fit.get_forecast(steps=max(horizons_list), exog=X_test.head(max(horizons_list)))
            forecast_values = forecasts.predicted_mean
            for h in horizons_list:
                y_true_h, y_pred_h = y_test.head(h).values, forecast_values.head(h).values
                if len(y_true_h) < h: continue
                mae = mean_absolute_error(y_true_h, y_pred_h)
                rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
                mase = calculate_mase(y_true_h, y_pred_h, y_train.values)
                accumulated_metrics[h]['maes'].append(mae)
                accumulated_metrics[h]['rmses'].append(rmse)
                if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        except Exception:
            continue

        # --- LÓGICA DE PLOTEO MODIFICADA ---
        # Si hay parámetros de ploteo Y es el primer fold Y el modelo se ajustó correctamente
        if plot_params and fold == 0 and results_fit:
            # Iterar sobre cada horizonte para generar un gráfico
            for h_plot in horizons_list:
                # Actualizar el horizonte en los parámetros del plot
                plot_params_horizon = plot_params.copy()
                plot_params_horizon['horizon'] = h_plot
                
                generate_sarimax_forecasting_plots(
                    df_full_imputed=df_city_data_input, 
                    df_full_ground_truth=df_city_data_ground_truth,
                    model_results=results_fit, 
                    original_nan_indices=original_nan_indices, 
                    fold_num=fold + 1, 
                    **plot_params_horizon
                )

    for h in horizons_list:
        results_by_horizon.append({'Horizon': h, 'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan, 'RMSE': np.nanmean(accumulated_metrics[h]['rmses']) if accumulated_metrics[h]['rmses'] else np.nan, 'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan})
    return results_by_horizon

def process_task(task_params):
    method_name = task_params['method_name']
    city_name = task_params['city_name']
    original_data_path = task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce')
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    city_name_safe = city_name.replace(" ", "_").replace("-", "_")
    order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
    try:
        with open(order_file_path, 'r') as f: orders = json.load(f)
        order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
    except Exception:
        order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.empty: return []
        if df_city_input.isnull().values.any():
            df_city_input.ffill(inplace=True); df_city_input.bfill(inplace=True)
            if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_sarimax_multi_horizon(df_city_input, df_city_input, set(), TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
        results = [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]
        return results

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    all_files = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))
    if not all_files: return []
    rep_file_path = all_files[0]
    file_name = os.path.basename(rep_file_path)
    rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
    except Exception: return []
    if df_rep.isnull().values.any(): df_rep = df_rep.ffill().bfill()
    if df_rep.isnull().values.any(): return []
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
    original_nan_indices = set()
    try:
        df_with_nans = pd.read_csv(os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados', city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
    except FileNotFoundError: pass
    horizon_metrics_rep = train_predict_eval_sarimax_multi_horizon(df_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
    results = [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics_rep]
    return results

if __name__ == '__main__':
    script_start_time = time.time()
    
    if not os.path.exists(ORIGINAL_DATA_FILE):
        print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'."); exit()
    print(f"Archivo de datos original encontrado en: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()

    tasks_params = []
    for city_name in cities_to_process:
        tasks_params.append({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE})

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks_params.append({
                        'method_name': method_name, 'city_name': city_name, 
                        'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE
                    })

    tasks = [delayed(process_task)(params) for params in tasks_params]
    
    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    print("Iniciando procesamiento...")
    
    results_list = Parallel(n_jobs=-1, backend="loky")(
        tqdm(tasks, desc="Progreso General de Tareas")
    )
    
    print("\nProcesamiento en paralelo completado. Recolectando y guardando resultados...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        print("Creando DataFrame principal y guardando métricas detalladas...")
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_output_path, index=False)
        
        print("Calculando y guardando resúmenes...")
        df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'), Count=('MAE', 'count')).reset_index()
        summary_by_city_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
        df_summary_by_city.to_csv(summary_by_city_path, index=False)

        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
        df_global_summary.to_csv(global_summary_path, index=False)
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        avg_over_horizons_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
        df_avg_over_horizons.to_csv(avg_over_horizons_path, index=False)
        
        print("\n--- Resumen Global por Método y Horizonte ---\n", df_global_summary.to_string())
        print("\n--- Resumen Final por Método (promedio de todas las ciudades, escenarios y horizontes) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados para analizar.")

    script_end_time = time.time()
    print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")
    print(f"Gráficos de ejemplo guardados en: {PREDICTION_PLOTS_OUTPUT_DIR}")

Random Forest - EJECUTAR CON TODO

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- PARÁMETROS DE EJECUCIÓN ---
N_LAGS_AQI = 120
N_LAGS_EXO = 120
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'RandomForest_full'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def create_multivariate_lagged_features_rf(df, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizon):
    features = []
    for i in range(horizon, n_lags_target + horizon):
        features.append(df[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col != target_col:
            for i in range(horizon, n_lags_exo + horizon):
                features.append(df[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_cols:
        if tf_col in df.columns:
            features.append(df[tf_col])
    return pd.concat(features, axis=1)

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

# --- NUEVA FUNCIÓN PARA ENCONTRAR BLOQUES DE NANS ---
def find_nan_blocks(original_nan_indices):
    if not original_nan_indices:
        return []
    nan_list = sorted(list(original_nan_indices))
    if not nan_list:
        return []
    blocks = []
    start_block = nan_list[0]
    for i in range(1, len(nan_list)):
        if nan_list[i] != nan_list[i-1] + 1:
            blocks.append((start_block, nan_list[i-1]))
            start_block = nan_list[i]
    blocks.append((start_block, nan_list[-1]))
    return blocks

def generate_rf_forecasting_plots(df_imputed_data, df_ground_truth_data, model, X_all_features_with_nans, nan_blocks, city, scenario, method, rep_id, horizon, output_dir, n_examples=1, history_len=300):
    max_lags_h = max(N_LAGS_AQI, N_LAGS_EXO) + horizon
    possible_points = range(max_lags_h + history_len, len(df_imputed_data) - horizon - 1)
    
    valid_points = []
    if not nan_blocks or method == 'Original_Data_Baseline':
        valid_points = list(possible_points)
    else:
        for p in possible_points:
            history_start, history_end = p - max_lags_h, p - horizon
            future_indices = range(p, p + horizon)
            # Condición 1: Ningún NaN en el futuro
            if not any(block_start <= i <= block_end for i in future_indices for block_start, block_end in nan_blocks):
                 # Condición 2: Al menos un bloque de NaNs COMPLETAMENTE contenido en el historial
                if any(block_start >= history_start and block_end <= history_end for block_start, block_end in nan_blocks):
                    valid_points.append(p)

    if not valid_points: return
    example_indices = random.sample(valid_points, min(n_examples, len(valid_points)))
    
    for split_point in example_indices:
        try:
            plt.figure(figsize=(20, 8))
            future_gt_df = df_ground_truth_data.iloc[split_point:split_point + horizon]
            X_forecast_clean = X_all_features_with_nans.loc[future_gt_df.index].dropna()
            if X_forecast_clean.empty: plt.close(); continue
            
            predictions = model.predict(X_forecast_clean)
            preds_series = pd.Series(predictions, index=X_forecast_clean.index)
            
            plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
            plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
            plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
            plt.plot(preds_series.index, preds_series, label='Forecast', color='green', linestyle='--')
            
            plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
            plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
            plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
            
            scenario_safe = scenario.replace('/', '_').replace('\\', '_')
            plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
            os.makedirs(plot_save_dir, exist_ok=True)
            plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
            plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
            plt.close()
        except Exception:
            plt.close()

def train_predict_eval_rf_multi_horizon(df_imputed, df_ground_truth, nan_blocks, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizons, plot_params=None):
    results_by_horizon = []
    
    for horizon in tqdm(horizons, desc="    Horizons", leave=False):
        X_with_nans = create_multivariate_lagged_features_rf(df_imputed, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizon)
        y = df_ground_truth[target_col]

        train_data = pd.concat([X_with_nans, y], axis=1).dropna()
        X_train_clean, y_train_clean = train_data.drop(columns=[target_col]), train_data[target_col]

        if X_train_clean.empty or len(X_train_clean) < 200:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=1)
        model.fit(X_train_clean, y_train_clean)
        
        if plot_params:
            plot_params_h = plot_params.copy()
            plot_params_h['horizon'] = horizon
            generate_rf_forecasting_plots(df_imputed, df_ground_truth, model, X_with_nans, nan_blocks, **plot_params_h)
            
        max_lags_h = max(n_lags_target, n_lags_exo) + horizon
        predictable_indices = X_with_nans.dropna().index
        
        impact_indices = []
        if not nan_blocks or plot_params.get('method') == 'Original_Data_Baseline':
            impact_indices = predictable_indices.tolist()
        else:
            for idx in predictable_indices:
                history_start, history_end = idx - max_lags_h, idx - horizon
                if any(block_start >= history_start and block_end <= history_end for block_start, block_end in nan_blocks):
                    impact_indices.append(idx)
        
        if not impact_indices:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue
            
        X_eval = X_with_nans.loc[impact_indices]
        y_eval_true = y.loc[impact_indices]
        predictions = model.predict(X_eval)
        
        mae = mean_absolute_error(y_eval_true, predictions)
        rmse = np.sqrt(mean_squared_error(y_eval_true, predictions))
        mase = calculate_mase(y_eval_true.values, predictions, y_train_clean)
        
        results_by_horizon.append({'Horizon': horizon, 'MAE': mae, 'RMSE': rmse, 'MASE': mase})
        
    return results_by_horizon

def process_rf_task(task_params):
    method_name = task_params['method_name']
    city_name = task_params['city_name']
    original_data_path = task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_rf_multi_horizon(df_city_input, df_city_input, [], TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, plot_params=plot_params)
        return [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    try:
        rep_file_path = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[0]
    except IndexError: return []

    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
    except Exception: return []
    
    rep_file_id = os.path.basename(rep_file_path)
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    
    nan_blocks = []
    try:
        nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
        df_with_nans = pd.read_csv(os.path.join(SIMULATED_DATA_ROOT_DIR, city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
        nan_blocks = find_nan_blocks(original_nan_indices)
    except FileNotFoundError: pass

    horizon_metrics = train_predict_eval_rf_multi_horizon(df_rep, df_city_ground_truth, nan_blocks, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, plot_params=plot_params)
    return [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

if __name__ == '__main__':
    script_start_time = time.time()
    if not os.path.exists(ORIGINAL_DATA_FILE): exit()
    print(f"Archivo de datos original encontrado: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()
    tasks = []
    for city_name in cities_to_process:
        tasks.append(delayed(process_rf_task)({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE}))

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks.append(delayed(process_rf_task)({'method_name': method_name, 'city_name': city_name, 'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE}))

    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    results_list = Parallel(n_jobs=-1, backend="loky")(tqdm(tasks, desc="Procesando Tareas RF"))
    
    print("\nProcesamiento completado. Generando resúmenes...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        df_all_metrics.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("\n[OK] Métricas detalladas guardadas.")
        
        df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary_by_city.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad guardado.")

        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global guardado.")
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final guardado.")
        
        print("\n--- Resumen Final (promedio de todo) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados.")

    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

LSTM

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
from keras import layers, Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- CONFIGURACIÓN DE GPU ---
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

# --- PARÁMETROS DE CONFIGURACIÓN Y MODELO ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_LAGS = 120
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# Parámetros de ejecución y ploteo
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300
N_WFV_FOLDS = 3 

# Hiperparámetros del modelo LSTM
LSTM_UNITS = 64
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
CLIPNORM = 1.0
EPOCHS = 50
BATCH_SIZE = 128

# --- NUEVO: Hiperparámetros para Callbacks ---
ES_PATIENCE = 15  # Paciencia para Early Stopping
LR_PATIENCE = 7   # Paciencia para reducir el Learning Rate


# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'LSTM_full'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
MODELS_TEMP_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'temp_models')
for d in [PREDICTION_METRICS_OUTPUT_DIR, PREDICTION_PLOTS_OUTPUT_DIR, MODELS_TEMP_DIR]: os.makedirs(d, exist_ok=True)

# --- FUNCIONES AUXILIARES ---
def create_sequences_single_shot(input_data, target_data, n_lags, horizon):
    X, y, indices = [], [], []
    y_multistep = np.lib.stride_tricks.as_strided(
        target_data,
        shape=(len(target_data) - horizon + 1, horizon),
        strides=(target_data.strides[0],) * 2
    )
    end_idx = len(input_data) - n_lags - horizon + 1
    for i in range(end_idx):
        X.append(input_data[i:(i + n_lags)])
    X = np.array(X)
    y = y_multistep[n_lags:n_lags + len(X)]
    indices = np.arange(n_lags, n_lags + len(X))
    return X, y, indices

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_lstm_forecasting_plots_single_shot(df_imputed_data, df_ground_truth_data, model, scaler_all, scaler_target, city, scenario, method, rep_id, output_dir, history_len=300):
    try:
        max_horizon = max(PREDICTION_HORIZONS)
        split_point = len(df_imputed_data) - history_len - max_horizon
        if split_point < N_LAGS: return
        all_cols = EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE]
        
        history_df = df_imputed_data.iloc[split_point - N_LAGS : split_point]
        input_seq = scaler_all.transform(history_df[all_cols])
        input_seq = input_seq.reshape(1, N_LAGS, input_seq.shape[1])
        prediction_scaled = model.predict(input_seq, verbose=0)
        prediction_unscaled = scaler_target.inverse_transform(prediction_scaled).flatten()
        
        for horizon in PREDICTION_HORIZONS:
            plt.figure(figsize=(20, 8))
            future_gt_df = df_ground_truth_data.iloc[split_point : split_point + horizon]
            future_pred = prediction_unscaled[:horizon]
            plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
            plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
            plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
            plt.plot(future_gt_df.index, future_pred, label='Forecast', color='green', linestyle='--')
            plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
            plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
            plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
            scenario_safe = scenario.replace('/', '_').replace('\\', '_')
            plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
            os.makedirs(plot_save_dir, exist_ok=True)
            plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
            plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
            plt.close()
    except Exception:
        plt.close()

def train_predict_eval_lstm_wfv_single_shot(df_imputed, df_ground_truth, original_nan_indices, target_col, all_features_cols, n_lags, horizons, plot_params=None):
    output_horizon = max(horizons)
    scaler_all = MinMaxScaler(); scaled_data = scaler_all.fit_transform(df_imputed[all_features_cols])
    scaler_target = MinMaxScaler(); scaled_target = scaler_target.fit_transform(df_imputed[[target_col]])
    
    train_val_size = int(len(scaled_data) * 0.8)
    train_val_data = scaled_data[:train_val_size]
    test_data = scaled_data[train_val_size:]
    
    tf.keras.backend.clear_session()
    X_tv, y_tv, _ = create_sequences_single_shot(train_val_data, scaled_target.flatten(), n_lags, output_horizon)
    
    if len(X_tv) < 200: return []
    
    tscv = TimeSeriesSplit(n_splits=N_WFV_FOLDS)
    best_val_loss = float('inf')
    model_path = os.path.join(MODELS_TEMP_DIR, f"best_model_{plot_params['city']}_{plot_params['method']}_{plot_params['scenario']}.keras")

    for fold, (train_idx, val_idx) in enumerate(tqdm(tscv.split(X_tv), total=N_WFV_FOLDS, desc="    WFV Folds", leave=False)):
        X_train, y_train = X_tv[train_idx], y_tv[train_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]
        
        inputs = layers.Input(shape=(X_train.shape[1], X_train.shape[2]))
        lstm_1 = layers.LSTM(LSTM_UNITS, return_sequences=True)(inputs); lstm_1 = layers.Dropout(DROPOUT_RATE)(lstm_1)
        lstm_2 = layers.LSTM(LSTM_UNITS // 2)(lstm_1); lstm_2 = layers.Dropout(DROPOUT_RATE)(lstm_2)
        outputs = layers.Dense(output_horizon)(lstm_2)
        model = Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM)
        model.compile(optimizer=optimizer, loss='mse')
        
        # --- NUEVOS CALLBACKS PARA MONITORIZACIÓN ---
        es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True, verbose=0)
        lr_scheduler_callback = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=LR_PATIENCE, factor=0.5, min_lr=1e-6, verbose=0)
        print_callback = tf.keras.callbacks.LambdaCallback(on_epoch_end=lambda epoch, logs: print(f" - Epoch {epoch+1:02d}: val_loss: {logs['val_loss']:.4f}", end=""))
        
        print(f"\n      Fold {fold+1}: Entrenando...")
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es_callback, lr_scheduler_callback, print_callback], verbose=0)
        
        val_loss = min(history.history['val_loss'])
        print(f"\n      Fold {fold+1} completado. Mejor val_loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print("      -> Nuevo mejor modelo encontrado. Guardando...")
            model.save(model_path)
    
    if not os.path.exists(model_path): return []
    best_model = tf.keras.models.load_model(model_path)
    if plot_params:
        generate_lstm_forecasting_plots_single_shot(df_imputed, df_ground_truth, best_model, scaler_all, scaler_target, **plot_params)
        
    X_test, y_test, indices_test = create_sequences_single_shot(test_data, scaled_target.flatten(), n_lags, output_horizon)
    if len(X_test) == 0: return []
    
    impact_indices = []
    if not original_nan_indices or plot_params.get('method') == 'Original_Data_Baseline':
        impact_indices = list(range(len(X_test)))
    else:
        for i, original_idx in enumerate(indices_test):
            history_window = range(original_idx + train_val_size - n_lags, original_idx + train_val_size)
            if any(h_idx in original_nan_indices for h_idx in history_window):
                impact_indices.append(i)
    
    if not impact_indices: return []

    X_eval = X_test[impact_indices]
    y_eval_true_scaled = y_test[impact_indices]
    predictions_scaled = best_model.predict(X_eval, verbose=0)
    
    results_by_horizon = []
    y_train_unscaled = scaler_target.inverse_transform(y_tv[train_idx])
    for h in horizons:
        pred_h_unscaled = scaler_target.inverse_transform(predictions_scaled[:, :h])
        true_h_unscaled = scaler_target.inverse_transform(y_eval_true_scaled[:, :h])
        mae = np.mean(np.abs(true_h_unscaled - pred_h_unscaled))
        rmse = np.sqrt(np.mean((true_h_unscaled - pred_h_unscaled)**2))
        mase = calculate_mase(true_h_unscaled.flatten(), pred_h_unscaled.flatten(), y_train_unscaled.flatten())
        results_by_horizon.append({'Horizon': h, 'MAE': mae, 'RMSE': rmse, 'MASE': mase})
        
    return results_by_horizon

def process_lstm_task(task_params):
    method_name, city_name, original_data_path = task_params['method_name'], task_params['city_name'], task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_lstm_wfv_single_shot(df_city_input, df_city_input, set(), TARGET_VARIABLE, EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE], N_LAGS, PREDICTION_HORIZONS, plot_params=plot_params)
        return [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    try: rep_file_path = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[0]
    except IndexError: return []
    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
        if df_rep.isnull().values.any(): df_rep = df_rep.ffill().bfill()
        if df_rep.isnull().values.any(): return []
    except Exception: return []
    
    rep_file_id = os.path.basename(rep_file_path)
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
    original_nan_indices = set()
    try:
        df_with_nans = pd.read_csv(os.path.join(SIMULATED_DATA_ROOT_DIR, city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
    except FileNotFoundError: pass

    horizon_metrics = train_predict_eval_lstm_wfv_single_shot(df_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE], N_LAGS, PREDICTION_HORIZONS, plot_params=plot_params)
    return [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

if __name__ == '__main__':
    script_start_time = time.time()
    if not os.path.exists(ORIGINAL_DATA_FILE): exit()
    print(f"Archivo de datos original encontrado: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()
    tasks_params = []
    for city_name in cities_to_process:
        tasks_params.append({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE})

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks_params.append({'method_name': method_name, 'city_name': city_name, 'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE})

    tasks = [delayed(process_lstm_task)(params) for params in tasks_params]
    
    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    results_list = Parallel(n_jobs=1)(tqdm(tasks, desc="Procesando Tareas LSTM"))
    
    print("\nProcesamiento completado. Generando resúmenes...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        df_all_metrics.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("\n[OK] Métricas detalladas guardadas.")
        
        df_summary = df_all_metrics.groupby(['Imputation_Method', 'City', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad y escenario guardado.")
        
        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global guardado.")
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final guardado.")
        
        print("\n--- Resumen Final (promedio de todo) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados.")

    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

MOMENT_PRE UNIVARIADO

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import time
import glob
import warnings
import traceback

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros de Predicción ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)

# Parámetros de Ejecución
BATCH_SIZE_PREDICT = 64 # Podemos usar un batch size más grande en CPU

# --- Filtros para Pruebas Rápidas ---
TARGET_CITY_FILTER = "Brasilia"
N_REPETITIONS_TO_PROCESS = 1

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_PRE_Pred_Univar' # Nuevo nombre para reflejar el enfoque
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)


# --- Dataset para Predicción Univariada ---
class MomentPredictionDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length = context_length
        self.max_horizon = max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self):
        return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        ground_truth_forecast = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(ground_truth_forecast, dtype=torch.float32)}

# --- Funciones Auxiliares de Métrica ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    y_train_no_nan = y_train_for_naive.dropna()
    if len(y_train_no_nan) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(y_train_no_nan.values)))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

# --- Función Principal de Predicción (simplificada a train/test split) ---
def predict_and_evaluate_moment_univar(df_city_data, target_col, model_pipeline, device, context_length, horizons, batch_size):
    series_data = df_city_data[target_col].values
    
    # Usar una división simple train/test en lugar de CV
    test_size = int(len(series_data) * 0.2)
    min_len_test = context_length + max(horizons)
    if test_size < min_len_test:
        print(f"      Datos insuficientes para la predicción. Test size ({test_size}) < min requerido ({min_len_test}).")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
        
    train_data = series_data[:-test_size]
    test_data = series_data[-test_size:]
    
    prediction_dataset = MomentPredictionDataset(test_data, context_length, max(horizons))
    if len(prediction_dataset) == 0:
        print("      No se pudieron crear secuencias de predicción."); return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
        
    prediction_loader = DataLoader(prediction_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    all_predictions, all_ground_truths = [], []
    
    model_pipeline.eval()
    with torch.no_grad():
        for batch in tqdm(prediction_loader, desc="      Prediciendo", leave=False):
            # La entrada es (batch, 1, context_len)
            context_batch = batch['context'].unsqueeze(1).to(device)
            try:
                outputs = model_pipeline(x_enc=context_batch, forecast_len=max(horizons))
                forecasts = outputs.forecast.cpu().squeeze(1)
                all_predictions.append(forecasts)
                all_ground_truths.append(batch['ground_truth_forecast'])
            except Exception as e:
                print(f"        Error en el forward pass del modelo: {e}"); continue
    
    if not all_predictions: return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}

    predictions_np = torch.cat(all_predictions).numpy()
    ground_truths_np = torch.cat(all_ground_truths).numpy()
    results = {}
    for h in horizons:
        y_pred, y_true = predictions_np[:, h-1], ground_truths_np[:, h-1]
        results[h] = {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MASE': calculate_mase(y_true, y_pred, pd.Series(train_data))
        }
    return results

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Predicción con MOMENT (Pre-entrenado, Univariado) ---"); print(f"Usando dispositivo: {device}")
try:
    moment_predictor = MOMENTPipeline.from_pretrained(
        MODEL_NAME, 
        model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON}
    )
    moment_predictor.init(); moment_predictor = moment_predictor.to(device)
    print("Modelo MOMENT (Pre-entrenado) para predicción cargado e inicializado.")
except Exception as e:
    print(f"Error crítico cargando modelo de predicción: {e}"); traceback.print_exc(); exit()

# --- 1. Procesar Baseline ---
try:
    df_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    cities_to_process = [TARGET_CITY_FILTER] if TARGET_CITY_FILTER else df_original['City'].unique()
    print("\n--- Procesando Baseline ---")
    for city in cities_to_process:
        if city not in df_original['City'].unique(): continue
        print(f"  Ciudad: {city}")
        df_city_data = df_original[df_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        if df_city_data.empty or df_city_data[TARGET_VARIABLE].isnull().any(): continue
        
        horizon_metrics = predict_and_evaluate_moment_univar(df_city_data, TARGET_VARIABLE, moment_predictor, device, CONTEXT_LENGTH, PREDICTION_HORIZONS, BATCH_SIZE_PREDICT)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})
except FileNotFoundError:
    print(f"Archivo original no encontrado: {ORIGINAL_DATA_FILE}. Omitiendo Baseline.")

# --- 2. Procesar Datos Imputados ---
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders = [c for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c)) and (TARGET_CITY_FILTER is None or c == TARGET_CITY_FILTER.replace(" ", "_"))]

    for city_folder in city_folders:
        city_path, city_name = os.path.join(imputed_csvs_root, city_folder), city_folder.replace("_", " ")
        print(f"  Ciudad: {city_name}")

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                horizon_metrics = predict_and_evaluate_moment_univar(df_rep, TARGET_VARIABLE, moment_predictor, device, CONTEXT_LENGTH, PREDICTION_HORIZONS, BATCH_SIZE_PREDICT)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

MOMENT_FT UNIVARIADO

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import traceback
import time
import glob
import warnings
import joblib
import sys

warnings.filterwarnings("ignore")

original_stdout = sys.stdout
original_stderr = sys.stderr
log_file = None
log_file_path = ""

# --- 1. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
# SOLUCIÓN: Restaurar los horizontes de predicción deseados
PREDICTION_HORIZONS = [12, 24, 36, 48, 60] 
MAX_HORIZON = max(PREDICTION_HORIZONS)

# --- Parámetros de Fine-Tuning ---
FINETUNE_EPOCHS = 10
BATCH_SIZE_FINETUNE = 32
LR = 1e-5
FINETUNE_TRAIN_SPLIT_RATIO = 0.7
ES_PATIENCE_FT = 15
MIN_DELTA_FT = 1e-6

# --- Parámetros de Predicción ---
N_CV_SPLITS_PREDICT = 5

# --- Filtros para Pruebas ---
TARGET_CITY_FILTER = "Brasilia"
N_REPETITIONS_TO_PROCESS = 5

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_FT_Pred_Univar'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
MODELS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'modelos')
SCALERS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'scalers')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR, exist_ok=True)

# --- 2. Clases y Funciones ---
class MomentForecastingDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length = context_length
        self.max_horizon = max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        ground_truth_forecast = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(ground_truth_forecast, dtype=torch.float32)}

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_moment_forecasting_model(model, train_loader, device, epochs, variable_name, model_save_path, max_horizon, learning_rate):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"  -> Iniciando fine-tuning para '{variable_name}' en {device.upper()} ({epochs} épocas máx)...")
    for epoch in range(epochs):
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"    Epoch {epoch+1}/{epochs}", leave=False)
        for batch_data in pbar:
            context = batch_data['context'].unsqueeze(1).to(device)
            ground_truth = batch_data['ground_truth_forecast'].unsqueeze(1).to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=context, forecast_len=max_horizon)
                loss = criterion(outputs.forecast, ground_truth)
                if torch.isnan(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())
            except Exception as e: continue
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
        print(f"    Epoch {epoch+1} Loss: {avg_epoch_loss:.6f}")
        if avg_epoch_loss < best_loss - MIN_DELTA_FT:
            best_loss = avg_epoch_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE_FT:
            print(f"    -> Early stopping en epoch {epoch+1}.")
            break
    print(f"  -> Fine-tuning para '{variable_name}' completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path):
        model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def predict_and_evaluate_moment_cv(df_city_data, target_col, model_pipeline, device, context_length, horizons, scaler, n_cv_splits):
    model_pipeline.eval()
    series_data = df_city_data[target_col]
    
    min_len = context_length + max(horizons) + n_cv_splits 
    if len(series_data) < min_len:
        print(f"      Datos insuficientes para la predicción con CV. Mínimo requerido: ~{min_len}, disponible: {len(series_data)}.")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
    
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons}

    print(f"      -> Iniciando evaluación con {n_cv_splits} fold(s) de CV...")
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data)):
        train_series = series_data.iloc[train_idx]
        context_unscaled = train_series.tail(context_length).values
        if len(context_unscaled) < context_length: continue

        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=max(horizons))
                predictions_scaled = outputs.forecast.cpu().squeeze().numpy()
                # SOLUCIÓN: Asegurar que la predicción sea al menos un array 1D
                predictions_scaled = np.atleast_1d(predictions_scaled)
            except Exception as e:
                print(f"        Error en predicción del Fold {fold+1}: {e}"); continue
        
        for h in horizons:
            # Ahora el indexado [h-1] es seguro
            y_pred_h_scaled = predictions_scaled[h-1]
            y_true_h_unscaled = series_data.iloc[test_idx[h-1]]
            y_pred_h_unscaled = scaler.inverse_transform(np.array([[y_pred_h_scaled]]))[0, 0]
            
            mae = np.abs(y_true_h_unscaled - y_pred_h_unscaled)
            rmse = (y_true_h_unscaled - y_pred_h_unscaled)**2
            mase = calculate_mase(np.array([y_true_h_unscaled]), np.array([y_pred_h_unscaled]), train_series)

            accumulated_metrics[h]['maes'].append(mae)
            accumulated_metrics[h]['rmses'].append(rmse)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)

    results = {}
    for h in horizons:
        results[h] = {
            'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan,
            'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])) if accumulated_metrics[h]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan
        }
    return results

# --- 3. Script Principal ---
all_prediction_metrics_detailed = []
script_start_time_pred = time.time()
training_device = "cuda" if torch.cuda.is_available() else "cpu"
prediction_device = "cpu"
print(f"--- Predicción con MOMENT (Fine-Tuned) ---\nDispositivo de entrenamiento: {training_device.upper()} | Dispositivo de predicción: {prediction_device.upper()}")

try:
    df_full_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    if TARGET_CITY_FILTER:
        cities_to_process = [TARGET_CITY_FILTER]
        print(f"\nATENCIÓN: Se procesará únicamente la ciudad: {TARGET_CITY_FILTER}")
    else:
        cities_to_process = df_full_original['City'].unique()
        print("\nATENCIÓN: Se procesarán todas las ciudades encontradas.")
except FileNotFoundError:
    print(f"Error crítico: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

finetuned_models, scalers = {}, {}
for city in cities_to_process:
    print(f"\n--- Iniciando Fine-Tuning para la ciudad: {city} ---")
    df_city_clean = df_full_original[df_full_original['City'] == city].copy()
    df_city_clean[TARGET_VARIABLE] = df_city_clean[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
    if df_city_clean.empty or df_city_clean[TARGET_VARIABLE].isnull().any(): print(f"  No hay datos limpios para {city}. Omitiendo."); continue

    scaler = StandardScaler()
    train_size = int(len(df_city_clean) * FINETUNE_TRAIN_SPLIT_RATIO)
    train_data_scaled = scaler.fit_transform(df_city_clean[TARGET_VARIABLE].values[:train_size].reshape(-1, 1)).flatten()
    
    ft_dataset = MomentForecastingDataset(train_data_scaled, CONTEXT_LENGTH, MAX_HORIZON)
    ft_loader = DataLoader(ft_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
    
    if len(ft_loader) == 0: print(f"  No se pudieron crear lotes de entrenamiento para {city}. Omitiendo."); continue
    
    try:
        moment_predictor_ft = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON})
        moment_predictor_ft.init(); moment_predictor_ft = moment_predictor_ft.to(training_device)
        
        city_safe_name = city.replace(" ", "_")
        model_save_path = os.path.join(MODELS_DIR, f"moment_forecaster_ft_{city_safe_name}.pth")
        scaler_save_path = os.path.join(SCALERS_DIR, f"scaler_{city_safe_name}.joblib")
        
        finetuned_model = train_moment_forecasting_model(moment_predictor_ft, ft_loader, training_device, FINETUNE_EPOCHS, TARGET_VARIABLE, model_save_path, MAX_HORIZON, LR)
        finetuned_models[city] = finetuned_model
        scalers[city] = scaler
        joblib.dump(scaler, scaler_save_path)
    except Exception as e:
        print(f"  Error crítico durante el fine-tuning para {city}: {e}"); traceback.print_exc()

print("\n--- Iniciando Fase de Evaluación ---")
# Baseline
print("\n--- Procesando Baseline (Original Interpolado) ---")
for city in cities_to_process:
    if city in finetuned_models:
        print(f"  Ciudad: {city}")
        df_city_data = df_full_original[df_full_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        
        model_to_eval = finetuned_models[city].to(prediction_device)
        scaler_to_eval = scalers[city]

        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

# Datos Imputados
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders_to_process = [c.replace("_", " ") for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c))]
    
    for city_name in cities_to_process:
        if city_name not in city_folders_to_process or city_name not in finetuned_models: continue
        print(f"  Ciudad: {city_name}")

        model_to_eval = finetuned_models[city_name].to(prediction_device)
        scaler_to_eval = scalers[city_name]
        
        city_path = os.path.join(imputed_csvs_root, city_name.replace(" ", "_"))
        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                horizon_metrics = predict_and_evaluate_moment_cv(df_rep, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- 4. Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time_pred)/60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if log_file and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

el mejor probablemente, con GPU

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import traceback
import time
import glob
import warnings
import joblib
import sys
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --- 1. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)

# --- Parámetros de Fine-Tuning ---
FINETUNE_EPOCHS = 2 # Reducido para pruebas rápidas
BATCH_SIZE_FINETUNE = 32
LR = 1e-4
FINETUNE_TRAIN_SPLIT_RATIO = 0.8
ES_PATIENCE_FT = 5
MIN_DELTA_FT = 1e-5

# --- Parámetros de Predicción ---
N_CV_SPLITS_PREDICT = 2 # Reducido para pruebas rápidas

# --- Filtros para Pruebas ---
TARGET_CITY_FILTER = None  # None para procesar todas las ciudades
N_REPETITIONS_TO_PROCESS = 2

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_FT_test'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
MODELS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'modelos')
SCALERS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'scalers')
PREDICTION_PLOTS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'prediction_plots')
PREDICTION_CSVS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'predictions_csvs')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_DIR, exist_ok=True)
os.makedirs(PREDICTION_CSVS_DIR, exist_ok=True)


# --- 2. Clases y Funciones ---
class MomentForecastingDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length, self.max_horizon = context_length, max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        target = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(target, dtype=torch.float32)}

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

def train_moment_forecasting_model(model, train_loader, device, epochs, variable_name, model_save_path, max_horizon, learning_rate):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"  -> Iniciando fine-tuning (linear probing) para '{variable_name}' en {device.upper()} ({epochs} épocas máx)...")
    for epoch in range(epochs):
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"    Epoch {epoch+1}/{epochs}", leave=False)
        for batch_data in pbar:
            context, ground_truth = batch_data['context'].unsqueeze(1).to(device), batch_data['ground_truth_forecast'].unsqueeze(1).to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=context, forecast_len=max_horizon)
                loss = criterion(outputs.forecast, ground_truth)
                if torch.isnan(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())
            except Exception: continue
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
        print(f"    Epoch {epoch+1} Loss: {avg_epoch_loss:.6f}")
        if avg_epoch_loss < best_loss - MIN_DELTA_FT:
            best_loss = avg_epoch_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE_FT:
            print(f"    -> Early stopping en epoch {epoch+1}."); break
    print(f"  -> Fine-tuning completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def save_prediction_results_to_csv(context_data, ground_truth_data, prediction_data, date_index, file_path):
    # SOLUCIÓN: Usar .values para ignorar el índice de la serie de fechas y crear uno nuevo.
    df_pred = pd.DataFrame({
        'date': date_index.values,
        'context': np.nan,
        'ground_truth': np.nan,
        'prediction': np.nan
    })
    
    context_len, horizon_len = len(context_data), len(ground_truth_data)
    df_pred.loc[:context_len-1, 'context'] = context_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'ground_truth'] = ground_truth_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'prediction'] = prediction_data
    
    df_pred.to_csv(file_path, index=False)

def plot_and_save_forecast(context, ground_truth, prediction, title, file_path):
    plt.figure(figsize=(15, 6))
    x_context = np.arange(len(context))
    x_forecast = np.arange(len(context), len(context) + len(ground_truth))
    plt.plot(x_context, context, label='Contexto (Datos de Entrada)', color='blue')
    plt.plot(x_forecast, ground_truth, label='Valores Reales (Ground Truth)', color='orange', linestyle='--')
    plt.plot(x_forecast, prediction, label='Predicción del Modelo', color='green', linestyle='-.')
    plt.title(title, fontsize=16)
    plt.xlabel('Pasos de Tiempo (Horas)')
    plt.ylabel('Valor de AQI')
    plt.legend(); plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout(); plt.savefig(file_path); plt.close()

def predict_and_evaluate_moment_cv(df_city_data, target_col, model_pipeline, device, context_length, horizons, scaler, n_cv_splits, **kwargs):
    model_pipeline.eval()
    series_data = df_city_data[target_col]
    min_len = context_length + max(horizons) + n_cv_splits 
    if len(series_data) < min_len:
        print(f"      Datos insuficientes para la predicción con CV. Mínimo requerido: ~{min_len}, disponible: {len(series_data)}.")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
    
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons}
    print(f"      -> Iniciando evaluación con {n_cv_splits} fold(s) de CV...")
    last_fold_data = {}
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data)):
        train_series = series_data.iloc[train_idx]
        context_unscaled = train_series.tail(context_length).values
        if len(context_unscaled) < context_length: continue
        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=max(horizons))
                predictions_scaled = np.atleast_1d(outputs.forecast.cpu().squeeze().numpy())
            except Exception as e: print(f"        Error en predicción del Fold {fold+1}: {e}"); continue
        
        ground_truth_unscaled = series_data.iloc[test_idx].values
        predictions_unscaled = scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

        for h in horizons:
            y_pred_h, y_true_h = predictions_unscaled[h-1], ground_truth_unscaled[h-1]
            mae, rmse = np.abs(y_true_h - y_pred_h), (y_true_h - y_pred_h)**2
            mase = calculate_mase(np.array([y_true_h]), np.array([y_pred_h]), train_series)
            accumulated_metrics[h]['maes'].append(mae); accumulated_metrics[h]['rmses'].append(rmse)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        
        if fold == n_cv_splits - 1:
            start_idx, end_idx = train_idx[-context_length], test_idx[-1]
            last_fold_data = {'context': context_unscaled, 'ground_truth': ground_truth_unscaled, 'prediction': predictions_unscaled, 'date_index': df_city_data['Date'].iloc[start_idx:end_idx+1]}

    if last_fold_data:
        plot_and_save_forecast(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], kwargs['plot_title'], kwargs['plot_path'])
        save_prediction_results_to_csv(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['csv_path'])
        print(f"      -> Gráfica y CSV de predicción del último fold guardados.")

    results = {}
    for h in horizons:
        results[h] = {'MAE': np.nanmean(accumulated_metrics[h]['maes']), 'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])), 'MASE': np.nanmean(accumulated_metrics[h]['mases'])}
    return results

# --- 3. Script Principal ---
all_prediction_metrics_detailed = []
script_start_time_pred = time.time()
training_device = "cuda" if torch.cuda.is_available() else "cpu"
prediction_device = "cpu"
print(f"--- Predicción con MOMENT (Fine-Tuned) ---\nDispositivo de entrenamiento: {training_device.upper()} | Dispositivo de predicción: {prediction_device.upper()}")

try:
    df_full_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    if TARGET_CITY_FILTER: cities_to_process = [TARGET_CITY_FILTER]
    else: cities_to_process = df_full_original['City'].unique()
    print(f"\nATENCIÓN: Se procesarán las siguientes ciudades: {', '.join(cities_to_process)}")
except FileNotFoundError:
    print(f"Error crítico: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

finetuned_models, scalers = {}, {}
for city in cities_to_process:
    print(f"\n--- Iniciando Fine-Tuning para la ciudad: {city} ---")
    df_city_clean = df_full_original[df_full_original['City'] == city].copy()
    df_city_clean[TARGET_VARIABLE] = df_city_clean[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
    if df_city_clean.empty or df_city_clean[TARGET_VARIABLE].isnull().any(): print(f"  No hay datos limpios para {city}. Omitiendo."); continue

    scaler = StandardScaler()
    train_size = int(len(df_city_clean) * FINETUNE_TRAIN_SPLIT_RATIO)
    train_data_scaled = scaler.fit_transform(df_city_clean[TARGET_VARIABLE].values[:train_size].reshape(-1, 1)).flatten()
    ft_dataset = MomentForecastingDataset(train_data_scaled, CONTEXT_LENGTH, MAX_HORIZON)
    ft_loader = DataLoader(ft_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
    
    if len(ft_loader) == 0: print(f"  No se pudieron crear lotes de entrenamiento para {city}. Omitiendo."); continue
    
    try:
        model_kwargs = {'task_name': 'forecasting', 
                        'forecast_horizon': MAX_HORIZON, 
                        'freeze_encoder': True, 
                        'freeze_embedder': True}
        moment_predictor_ft = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs=model_kwargs)
        moment_predictor_ft.init(); moment_predictor_ft = moment_predictor_ft.to(training_device)
        
        city_safe_name = city.replace(" ", "_")
        model_save_path = os.path.join(MODELS_DIR, f"moment_forecaster_ft_{city_safe_name}.pth")
        scaler_save_path = os.path.join(SCALERS_DIR, f"scaler_{city_safe_name}.joblib")
        
        finetuned_model = train_moment_forecasting_model(moment_predictor_ft, ft_loader, training_device, FINETUNE_EPOCHS, TARGET_VARIABLE, model_save_path, MAX_HORIZON, LR)
        finetuned_models[city] = finetuned_model
        scalers[city] = scaler
        joblib.dump(scaler, scaler_save_path)
    except Exception as e:
        print(f"  Error crítico durante el fine-tuning para {city}: {e}"); traceback.print_exc()

print("\n--- Iniciando Fase de Evaluación ---")
# Baseline
print("\n--- Procesando Baseline (Original Interpolado) ---")
for city in cities_to_process:
    if city in finetuned_models:
        print(f"  Ciudad: {city}")
        df_city_data = df_full_original[df_full_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        
        model_to_eval, scaler_to_eval = finetuned_models[city].to(prediction_device), scalers[city]
        
        plot_title = f'Predicción Baseline - {city}'
        plot_path = os.path.join(PREDICTION_PLOTS_DIR, f'baseline_{city.replace(" ", "_")}.png')
        csv_path = os.path.join(PREDICTION_CSVS_DIR, f'baseline_{city.replace(" ", "_")}.csv')

        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

# Datos Imputados
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders_to_process = [c.replace("_", " ") for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c))]
    
    for city_name in cities_to_process:
        if city_name not in city_folders_to_process or city_name not in finetuned_models: continue
        print(f"  Ciudad: {city_name}")

        model_to_eval, scaler_to_eval = finetuned_models[city_name].to(prediction_device), scalers[city_name]
        
        city_path = os.path.join(imputed_csvs_root, city_name.replace(" ", "_"))
        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                plot_title = f'Predicción - {city_name}\nMétodo: {method_name}\nEscenario: {scenario_folder} - {rep_id}'
                plot_path = os.path.join(PREDICTION_PLOTS_DIR, f'{method_name}_{city_name.replace(" ","_")}_{scenario_folder}_{rep_id}.png')
                csv_path = os.path.join(PREDICTION_CSVS_DIR, f'{method_name}_{city_name.replace(" ","_")}_{scenario_folder}_{rep_id}.csv')
                
                horizon_metrics = predict_and_evaluate_moment_cv(df_rep, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- 4. Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time_pred)/60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
if log_file and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")